# Mosaic Builder — Visual Decision Demo

This notebook walks through each stage of the pipeline with visual output so you can evaluate whether the decisions are appropriate for the final mosaic.

In [ ]:
# Reload mosaic_builder on every cell execution, so editing the library does not
# need a kernel restart. Caveats: values imported by name (`from x import CONST`)
# keep their old binding, and dataclass/schema changes still need a restart
# because objects already built in the kernel keep their old shape.
%load_ext autoreload
%autoreload 2

In [ ]:
import json
import sys
import tempfile
from pathlib import Path

import numpy as np
from IPython.display import HTML, display
from PIL import Image, ImageDraw

# The package is installed into the project's venv via `uv sync`, so a plain
# import should work. The sys.path insert below is only a fallback for
# running this notebook without that install.
try:
    import mosaic_builder  # noqa: F401
except ImportError:
    sys.path.insert(0, str(Path.cwd().parent / "src"))

from mosaic_builder.duckdb_store import (
    SchemaVersionError,
    ensure_schema,
    get_all_images,
    get_tile_lab_descriptors,
    get_tile_sides,
    get_tiles_for_image,
    open_database,
)
from mosaic_builder.ingest import ingest_gallery
from mosaic_builder.matcher import run_match, summarize
from mosaic_builder.renderer import assemble_mosaic, blend_with_target
from mosaic_builder.tiler import (
    color_contrast_score,
    delta_e,
    extract_character_tiles,
    fixture_likelihood_score,
    iter_gallery_images,
    load_image,
    mean_lab,
    periodicity_score,
    resize_tile,
    texture_score,
)

## Helper: side-by-side display

In [ ]:
import base64
import io


def img_to_data_url(img, max_side=None):
    if max_side and max(img.size) > max_side:
        img = img.copy()
        img.thumbnail((max_side, max_side))
    buf = io.BytesIO()
    img.save(buf, format="PNG")
    return "data:image/png;base64," + base64.b64encode(buf.getvalue()).decode()


def show_images(images, labels=None, max_side=200):
    """Display a row of images with optional labels."""
    html = '<div style="display:flex; gap:12px; align-items:flex-start; flex-wrap:wrap;">'
    for i, img in enumerate(images):
        url = img_to_data_url(img, max_side=max_side)
        label = labels[i] if labels else ""
        html += f'<div style="text-align:center"><img src="{url}" style="border:1px solid #444; border-radius:4px;"/>'
        if label:
            html += f'<div style="font-size:11px; color:#888; margin-top:4px;">{label}</div>'
        html += "</div>"
    html += "</div>"
    display(HTML(html))


def swatch(r, g, b, size=40):
    """Create a solid color swatch image."""
    return Image.new("RGB", (size, size), (int(r), int(g), int(b)))

## 1. Load gallery images

Two different things come out of this cell:

- **`test_images`** — the first 3 images, used for the close-up walkthrough of crop extraction and
  per-tile scoring in sections 2 through 5b. Three is plenty to *look* at in detail.
- **`DB_SOURCE_DIR`** — the directory section 6 ingests into the demo database. This is the
  **whole gallery**, so the matching and rendering sections downstream work against a realistic
  tile pool. Mosaic quality depends almost entirely on how many tiles the matcher can choose
  between, so building the demo database from a handful of images would misrepresent what the
  pipeline does — it would look repetitive no matter how good the matcher is.

`GALLERY_DIR` points at the `gallery/` symlink, which is not committed to the repo and will not
exist for most people who clone it. **If it is missing, this cell generates
`SYNTHETIC_IMAGE_COUNT` synthetic images with PIL instead** (and says so explicitly below) so the
rest of the notebook still runs end to end.

Set `DB_IMAGE_LIMIT` to an integer for a faster run at the cost of mosaic quality; `None` uses
every image available.

In [ ]:
GALLERY_DIR = Path.cwd().parent / "gallery"

#: How many images the close-up walkthrough (sections 2-5b) displays in detail.
DISPLAY_IMAGE_COUNT = 3
#: Images to synthesise when no real gallery is present. Enough that the
#: matcher has a realistic pool to choose from rather than a toy one.
SYNTHETIC_IMAGE_COUNT = 60
#: Cap on images ingested into the demo database; None means "all of them".
DB_IMAGE_LIMIT = None

gallery_paths = []
if GALLERY_DIR.exists():
    gallery_paths = list(iter_gallery_images(GALLERY_DIR))

if len(gallery_paths) >= DISPLAY_IMAGE_COUNT:
    DB_SOURCE_DIR = GALLERY_DIR
    print(f"Using the real gallery at {GALLERY_DIR}: {len(gallery_paths)} images")
else:
    print(
        f"NOTE: gallery not usable at {GALLERY_DIR} (found {len(gallery_paths)} image(s)) "
        f"-- generating {SYNTHETIC_IMAGE_COUNT} synthetic images with PIL instead so this "
        "notebook runs anywhere."
    )

    def make_synthetic_image(size, base_color, seed):
        """A solid-ish colour field with random ellipses, so texture_score has something to rank."""
        img = Image.new("RGB", size, base_color)
        draw = ImageDraw.Draw(img)
        rng = np.random.default_rng(seed)
        for _ in range(40):
            x0, y0 = int(rng.integers(0, size[0])), int(rng.integers(0, size[1]))
            w, h = int(rng.integers(10, 60)), int(rng.integers(10, 60))
            jitter = rng.integers(-80, 80, size=3)
            color = tuple(int(v) for v in np.clip(np.array(base_color) + jitter, 0, 255))
            draw.ellipse([x0, y0, x0 + w, y0 + h], fill=color)
        return img

    synth_dir = Path(tempfile.mkdtemp()) / "synthetic_gallery"
    synth_dir.mkdir()
    shapes = [(640, 360), (360, 640), (480, 480)]
    rng = np.random.default_rng(0)
    gallery_paths = []
    for i in range(SYNTHETIC_IMAGE_COUNT):
        base_color = tuple(int(v) for v in rng.integers(30, 226, size=3))
        img = make_synthetic_image(shapes[i % len(shapes)], base_color, seed=i + 1)
        path = synth_dir / f"synthetic_{i:03d}.png"
        img.save(path)
        gallery_paths.append(path)
    DB_SOURCE_DIR = synth_dir

# The close-up sections below only need a few images to look at in detail; the
# demo database built in section 6 uses DB_SOURCE_DIR, i.e. all of them.
display_paths = gallery_paths[:DISPLAY_IMAGE_COUNT]
test_images = {}
for p in display_paths:
    img = load_image(p)
    test_images[p.stem] = img

show_images(
    list(test_images.values()),
    [f"{name}\n{img.size[0]}\u00d7{img.size[1]}" for name, img in test_images.items()],
    max_side=250,
)
print(f"Displaying {len(test_images)} of {len(gallery_paths)} images in sections 2-5b")

## 2. Character tile extraction

For each image, `extract_character_tiles` finds up to 3 maximal square crops, ranked by texture score.

**Key decision:** We take the largest possible square (min of width, height) and slide it to 3 candidate positions (left/center/right or top/center/bottom). The crop with the most texture wins.

This means:
- Wide images get horizontal crops
- Tall images get vertical crops
- Square images get a single crop

In [ ]:
for name, img in test_images.items():
    tiles = extract_character_tiles(img, max_tiles=3)

    # Draw crop boxes on the source
    annotated = img.copy()
    draw = ImageDraw.Draw(annotated)
    colors = ["#ff4444", "#44ff44", "#4444ff"]
    for i, tc in enumerate(tiles):
        draw.rectangle(tc.box, outline=colors[i % 3], width=3)
        draw.text((tc.box[0] + 4, tc.box[1] + 4), f"#{i + 1}", fill=colors[i % 3])

    all_imgs = [annotated] + [tc.image for tc in tiles]
    all_labels = [f"{name} (source)\n{img.size[0]}×{img.size[1]}"] + [
        f"Tile #{i + 1}\nbox={tc.box}\nscore={tc.score:.1f}" for i, tc in enumerate(tiles)
    ]

    print(f"\n{'=' * 60}")
    print(f"{name}: {len(tiles)} tiles extracted")
    show_images(all_imgs, all_labels, max_side=200)

## 3. Resize and observe quality loss

Tiles get resized to a uniform `tile_side` (e.g. 64px). This is what actually gets stored in the DB and used for matching.

**Question to evaluate:** Does the resized tile still look representative of the source region?

In [ ]:
tile_sides = [32, 64, 128]
# Use the first gallery image
demo_name = list(test_images.keys())[0]
demo_img = test_images[demo_name]
tiles = extract_character_tiles(demo_img, max_tiles=1)
raw_tile = tiles[0].image

resized = [resize_tile(raw_tile, s) for s in tile_sides]
show_images(
    [raw_tile] + resized,
    [f"Original crop\n{raw_tile.size[0]}×{raw_tile.size[1]}"]
    + [f"tile_side={s}\n{s}×{s}px" for s in tile_sides],
    max_side=300,
)
print("\nSmaller tile_side = smaller file, faster matching, but less detail.")
print("Larger tile_side = more detail in final mosaic, but more storage + slower matching.")

## 4. RGB vs LAB color descriptors

Each tile stores mean RGB and mean LAB. LAB is perceptually uniform — equal distances in LAB space correspond to equal perceived color differences.

**Why this matters:** When the matcher picks a tile for a grid cell, we want the *perceived* color match to be good, not just the numerical RGB distance.

In [ ]:
# Compare RGB and LAB distances for color pairs that humans perceive differently
color_pairs = [
    ("Reds", (200, 30, 30), (220, 50, 50)),
    ("Red→Green", (200, 30, 30), (30, 200, 30)),
    ("Blues", (30, 30, 200), (50, 50, 220)),
    ("Gray tones", (100, 100, 100), (130, 130, 130)),
    ("Cyan↔Mag", (0, 200, 200), (200, 0, 200)),
]

print(f"{'Pair':<14} {'RGB dist':>10} {'ΔE (LAB)':>10}  Visual")
print("-" * 60)

for pair_name, c1, c2 in color_pairs:
    img1 = Image.new("RGB", (20, 20), c1)
    img2 = Image.new("RGB", (20, 20), c2)

    # Euclidean RGB distance
    rgb_dist = np.sqrt(sum((a - b) ** 2 for a, b in zip(c1, c2, strict=True)))

    # Delta-E LAB distance
    lab1 = mean_lab(img1)
    lab2 = mean_lab(img2)
    de = delta_e(lab1, lab2)

    print(f"{pair_name:<14} {rgb_dist:>10.1f} {de:>10.1f}")

# Show swatches
for pair_name, c1, c2 in color_pairs:
    print(f"\n{pair_name}:")
    show_images([swatch(*c1), swatch(*c2)], [str(c1), str(c2)], max_side=40)

## 5. Texture scoring

Tiles are ranked by texture score (variance + gradient magnitude). Higher scores = more visual interest.

**Decision to evaluate:** Does the scoring correctly prefer textured regions over flat ones?

In [ ]:
# Score all extracted tiles from our gallery images
all_scored = []
for name, img in test_images.items():
    tiles = extract_character_tiles(img, max_tiles=3)
    for i, tc in enumerate(tiles):
        resized = resize_tile(tc.image, 64)
        score = texture_score(resized)
        all_scored.append((f"{name}#{i + 1}", resized, score))

all_scored.sort(key=lambda x: x[2], reverse=True)

print("All tiles ranked by texture score (highest first):")
show_images(
    [img for _, img, _ in all_scored],
    [f"{name}\nscore={score:.1f}" for name, _, score in all_scored],
    max_side=100,
)

## 5b. Fixture-detection scoring (color contrast, periodicity, composite)

Beyond generic texture, three additional per-tile scores target a specific use case for this
gallery: telling apart "the fixture" (e.g. a urinal/toilet) from "the background behind it" in a
themed photo.

- `color_contrast_score`: Otsu-style bimodality over the tile's L* channel — high when the crop
  splits cleanly into two large, well-separated brightness regions (e.g. a dark object against a
  light wall); stays low for a thin minority of grout lines or a uniform white-on-white region.
- `periodicity_score`: FFT-based concentration of spectral energy into a few peaks — high for
  regular repeating structure (a grout grid), low for a smooth, non-periodic surface.
- `fixture_likelihood_score`: blends the two *continuously* — colour separation carries 70% of the
  evidence because it is the less ambiguous signal, non-periodicity the remaining 30% — then scales
  the result by a texture-derived **presence** factor, so a blank tile scores ~0 instead of reading
  as "smooth, therefore a fixture". Computed on read, never stored, so retuning its constants takes
  effect without re-ingesting.

**Two caveats when reading the numbers below.** The non-periodicity half only discriminates when the
background actually *is* patterned (tiled, grouted, brickwork) — against a plain painted wall it has
nothing to go on. And because colour carries most of the weight, a white-fixture-on-white-wall crop
tops out near 0.3: the score is confidence-weighted, so weak-evidence fixtures rank below
strong-evidence ones by design. Rough expectations: ~0.0 blank, ~0.07 tiled wall, ~0.29 white
fixture on white wall, ~0.56 clearly separated object, ~0.84 dark fixture on a light wall.

**Decision to evaluate:** Do high `fixture_likelihood_score` tiles actually look fixture-adjacent,
and do high `periodicity_score` tiles look like tiled/grid surfaces? The underlying constants
(`color_contrast_norm`, `texture_norm`, `color_weight`, `dc_radius_frac`, `top_k`) are first-guess
values — see `tiler.py` — not yet tuned against this gallery.

In [ ]:
all_fixture_scored = []
for name, img in test_images.items():
    tiles = extract_character_tiles(img, max_tiles=3)
    for i, tc in enumerate(tiles):
        resized = resize_tile(tc.image, 64)
        texture = texture_score(resized)
        cc = color_contrast_score(resized)
        per = periodicity_score(resized)
        fixture = fixture_likelihood_score(texture, cc, per)
        all_fixture_scored.append((f"{name}#{i + 1}", resized, fixture, cc, per))

all_fixture_scored.sort(key=lambda x: x[2], reverse=True)

print("All tiles ranked by fixture_likelihood_score (highest first):")
show_images(
    [img for _, img, _, _, _ in all_fixture_scored],
    [
        f"{name}\nfixture={fixture:.2f}\ncc={cc:.1f} per={per:.2f}"
        for name, _, fixture, cc, per in all_fixture_scored
    ],
    max_side=100,
)

## 6. Full ingest → DB round-trip

Two separate things happen here, because they have different needs:

1. **A live ingest demo** over a handful of images — enough to show the API working and what a
   stored tile looks like. Quick.
2. **Choosing the database everything downstream matches against.** Matching quality depends almost
   entirely on how many tiles are available, so this wants the whole gallery. If `mosaic.duckdb`
   already exists at the repo root it is reused as-is, since rebuilding an identical copy costs
   about two minutes per run for no benefit. Build it once with:

   ```
   uv run mosaic-builder ingest gallery --db mosaic.duckdb --tile-side 64
   ```

   Without it (or with `DB_IMAGE_LIMIT` set) this cell ingests from scratch instead — correct, just
   slower.

In [ ]:
import shutil

tile_side = 64
PREBUILT_DB = Path.cwd().parent / "mosaic.duckdb"
#: Images ingested live below, purely to demonstrate the ingest API.
INGEST_DEMO_IMAGES = 6

tmp = Path(tempfile.mkdtemp())


def _usable(path: Path) -> bool:
    """True when `path` is a readable database at the current schema version."""
    if not path.exists():
        return False
    conn = open_database(path)
    try:
        ensure_schema(conn)
        return bool(get_tile_sides(conn))
    except SchemaVersionError:
        return False
    finally:
        conn.close()


# 1. Live ingest demo on a small sample.
sample_dir = tmp / "sample_gallery"
sample_dir.mkdir()
for p in gallery_paths[:INGEST_DEMO_IMAGES]:
    shutil.copy2(p, sample_dir / p.name)
sample = ingest_gallery(sample_dir, tmp / "sample.duckdb", tile_side=tile_side, show_progress=False)
print(f"Ingest demo: {sample.images} images -> {sample.tiles} tiles")

# 2. Database for everything downstream.
if DB_IMAGE_LIMIT is None and _usable(PREBUILT_DB):
    db_path = PREBUILT_DB
    print(f"Matching against prebuilt {PREBUILT_DB.name} (skips a ~2 min rebuild).")
else:
    db_path = tmp / "demo.duckdb"
    if DB_IMAGE_LIMIT is None:
        ingest_dir = DB_SOURCE_DIR
    else:
        ingest_dir = tmp / "limited_gallery"
        ingest_dir.mkdir()
        for p in gallery_paths[:DB_IMAGE_LIMIT]:
            shutil.copy2(p, ingest_dir / p.name)
    print(f"Building a database from {ingest_dir} -- this is the slow step...")
    summary = ingest_gallery(ingest_dir, db_path, tile_side=tile_side, show_progress=False)
    print(f"Ingested {summary.images} images, {summary.tiles} tiles")

print(f"DB: {db_path}")

In [ ]:
conn = open_database(db_path)
ensure_schema(conn)

images = get_all_images(conn)
print(f"Database holds {len(images)} images; showing the first {DISPLAY_IMAGE_COUNT}.")

for img_row in images[:DISPLAY_IMAGE_COUNT]:
    tiles = get_tiles_for_image(conn, img_row["image_id"])

    # Load source for comparison. Older databases may hold paths relative to
    # wherever ingest ran, so fall back to resolving against the repo root.
    source_path = Path(img_row["path"])
    if not source_path.exists():
        source_path = Path.cwd().parent / source_path
    source = Image.open(source_path)

    # Decode stored tile PNGs
    tile_imgs = [Image.open(io.BytesIO(t["tile_png"])) for t in tiles]
    # fixture_likelihood_score is derived, not stored -- computed here from the
    # stored primitives so retuning its constants needs no re-ingest.
    tile_labels = [
        f"Tile #{t['tile_index'] + 1}\n"
        f"RGB=({t['mean_r']:.0f},{t['mean_g']:.0f},{t['mean_b']:.0f})\n"
        f"LAB=({t['mean_L']:.1f},{t['mean_a']:.1f},{t['mean_bb']:.1f})\n"
        f"score={t['score']:.1f} cov={t['coverage']:.2f}\n"
        f"fixture={fixture_likelihood_score(t['score'], t['color_contrast'], t['periodicity']):.2f} "
        f"cc={t['color_contrast']:.1f} per={t['periodicity']:.2f}"
        for t in tiles
    ]

    # Show mean-color swatches next to actual tiles
    swatch_imgs = [swatch(t["mean_r"], t["mean_g"], t["mean_b"]) for t in tiles]

    name = Path(img_row["path"]).stem
    print(f"\n{'=' * 60}")
    print(f"{name} ({img_row['width']}×{img_row['height']}) — {len(tiles)} tiles stored")

    # Row 1: source + tiles
    show_images(
        [source] + tile_imgs,
        [f"Source\n{img_row['width']}×{img_row['height']}"] + tile_labels,
        max_side=160,
    )

    # Row 2: mean-color swatches
    print("  Mean color swatches (what the matcher sees):")
    show_images(
        swatch_imgs,
        [f"Tile #{t['tile_index'] + 1} mean RGB" for t in tiles],
        max_side=50,
    )

conn.close()

## 7. Delta-E tile similarity matrix

How similar are the stored tiles to each other? This shows whether the gallery has enough colour
diversity for a good mosaic.

The full matrix is `n x n`, so for a real gallery it is far too large to print. The cell below
samples `MATRIX_SAMPLE` tiles for the printed matrix, then reports the mean nearest-neighbour
distance across **every** stored tile — that summary statistic, not the matrix, is the number that
actually predicts mosaic quality. Below roughly 2 Delta-E the gallery is tightly clustered and
near-duplicate substitutes are always available.

In [ ]:
MATRIX_SAMPLE = 9

conn = open_database(db_path)
descs = get_tile_lab_descriptors(conn)
conn.close()

lab_all = np.array([[d["mean_L"], d["mean_a"], d["mean_bb"]] for d in descs], dtype=np.float64)

# Printed matrix: a small evenly-spaced sample, so it stays readable.
step = max(1, len(descs) // MATRIX_SAMPLE)
sample_idx = list(range(0, len(descs), step))[:MATRIX_SAMPLE]
sample = lab_all[sample_idx]
matrix = np.sqrt(((sample[:, None, :] - sample[None, :, :]) ** 2).sum(-1))

tile_names = [f"img{descs[i]['image_id']}:t{descs[i]['tile_index']}" for i in sample_idx]
print(f"Delta-E between {len(sample_idx)} sampled tiles (lower = more similar):")
print(f"{'':>12}" + "".join(f"{name:>12}" for name in tile_names))
for i, name in enumerate(tile_names):
    print(f"{name:>12}" + "".join(f"{matrix[i, j]:>12.1f}" for j in range(len(sample_idx))))

# Summary over every tile, computed in chunks to bound memory.
nearest = np.empty(len(lab_all))
for start in range(0, len(lab_all), 512):
    chunk = lab_all[start : start + 512]
    d = np.sqrt(((chunk[:, None, :] - lab_all[None, :, :]) ** 2).sum(-1))
    d[np.arange(len(chunk)), np.arange(start, start + len(chunk))] = np.inf
    nearest[start : start + len(chunk)] = d.min(axis=1)

print(f"\nAcross all {len(descs)} tiles:")
print(f"  mean nearest-neighbour Delta-E: {nearest.mean():.2f}")
print(f"  median: {np.median(nearest):.2f}   max: {nearest.max():.2f}")
print("  (low values mean many near-duplicates, so diversity is cheap to buy)")

## 8. Full pipeline: matching + rendering

Now that the gallery is ingested, run the matching and rendering stages on a real target image:
`examples/target_images/target_8bit_checkered_floor_01.png`.

`run_match` divides the target into a grid of `grain`-sized cells, computes each cell's mean LAB
colour, and assigns a tile to each. `assemble_mosaic` pastes those tiles onto a canvas, and
`blend_with_target` can blend the result partway back toward the original to improve legibility at
a distance.

This runs against the full database built in section 6, so the result below reflects what the
pipeline actually produces rather than a toy approximation.

In [ ]:
TARGET_PATH = Path.cwd().parent / "examples" / "target_images" / "target_8bit_checkered_floor_01.png"
GRAIN = 64

target_img = load_image(TARGET_PATH)

result = run_match(
    TARGET_PATH,
    db_path,
    grain=GRAIN,
    tile_side=tile_side,
    show_progress=False,
)
print(summarize(result))

conn = open_database(db_path)
ensure_schema(conn)
mosaic_img = assemble_mosaic(result, conn, show_progress=False)
conn.close()

blended_img = blend_with_target(mosaic_img, target_img, alpha=0.3)

show_images(
    [target_img, mosaic_img, blended_img],
    [
        f"Target\n{target_img.size[0]}\u00d7{target_img.size[1]}",
        f"Mosaic (demo DB, grain={GRAIN})\n{mosaic_img.size[0]}\u00d7{mosaic_img.size[1]}",
        "Mosaic blended 30% toward target",
    ],
    max_side=300,
)

## 9. Diversity versus colour fidelity

`--diversity` runs 0 to 10 and expresses a *preference*, not a constraint. Each candidate tile is
costed as `delta_e + weight × penalty`, where the penalty grows with how often the tile has already
been placed — measured against its fair share, `cells / tiles` — and with how close those
placements are to the current cell. The cheapest tile wins, so a heavily-reused tile can still be
chosen when it is a markedly better colour match. The dial never forces an obviously wrong tile in
the name of variety.

Scaling by fair share is what makes a dial position mean the same thing on a small gallery as on a
large one. When cells outnumber tiles, repeats are unavoidable; charging for them from the first
repeat would swamp colour distance and make the choice effectively random.

At 0 this is plain nearest-colour matching, which leans hard on a few tiles. Turn it up and the
matcher spreads work across comparable alternatives, paying a little mean delta-E for a lot of
variety.

In [ ]:
settings = [
    {"label": "diversity=0 (pure colour match)", "diversity": 0.0},
    {"label": "diversity=3 (default)", "diversity": 3.0},
    {"label": "diversity=10 (max variety)", "diversity": 10.0},
]

mosaics = []
labels = []
for s in settings:
    r = run_match(
        TARGET_PATH,
        db_path,
        grain=GRAIN,
        tile_side=tile_side,
        diversity=s["diversity"],
        show_progress=False,
    )
    conn = open_database(db_path)
    ensure_schema(conn)
    m = assemble_mosaic(r, conn, show_progress=False)
    conn.close()
    mosaics.append(m)
    label = s["label"]
    labels.append(f"{label}\n{summarize(r)}")
    print(label, "->", summarize(r))

show_images(mosaics, labels, max_side=250)

## 10. Showcase: real gallery, real target

Every output above this point is stripped before committing (the notebook stays small in git), so
nothing above proves the pipeline works until you actually run it. This section is different: its
outputs — the mosaic, a side-by-side comparison, and a JSON sidecar with the exact parameters and
numbers — are committed to `examples/output/`, so the repository carries real proof without anyone
executing anything.

The target is a photo the user chose (a bathroom urinal), matched against the **entire** gallery
(not the 3-image demo database used above) at settings picked from the measured tradeoff table:
`--max-reuse 4 --min-repeat-dist 3`. Re-running this section requires the full gallery and its
ingested database (`mosaic.duckdb`); without it, this section prints a note and skips cleanly
instead of failing.


In [ ]:
ROOT = Path.cwd().parent

SHOWCASE_SOURCE = ROOT / "examples" / "target_images" / "target_urinal_01.avif"
SHOWCASE_TARGET = ROOT / "examples" / "target_images" / "target_urinal_01.png"
SHOWCASE_MAX_SIDE = 1024
SHOWCASE_CELLS = 30
SHOWCASE_DIVERSITY = 3.0
SHOWCASE_BLEND = 0.0
FULL_DB = ROOT / "mosaic.duckdb"
OUTPUT_DIR = ROOT / "examples" / "output"

MOSAIC_CEILING_BYTES = 7 * 1024 * 1024  # real photo mosaics compress poorly as PNG
COMPARISON_CEILING_BYTES = 1 * 1024 * 1024
TARGET_CEILING_BYTES = 2 * 1024 * 1024

In [ ]:
# Normalize the chosen target to a portable PNG. AVIF decoding depends on how Pillow
# was built, so every downstream cell (and the committed comparison image) uses this PNG,
# never the AVIF directly. This cell runs even without the gallery.
showcase_source_img = load_image(SHOWCASE_SOURCE)

showcase_target_img = showcase_source_img.copy()
if max(showcase_target_img.size) > SHOWCASE_MAX_SIDE:
    showcase_target_img.thumbnail((SHOWCASE_MAX_SIDE, SHOWCASE_MAX_SIDE), Image.Resampling.LANCZOS)

showcase_target_img.save(SHOWCASE_TARGET, format="PNG")

print(
    f"Normalized target: {SHOWCASE_SOURCE.name} "
    f"({showcase_source_img.size[0]}x{showcase_source_img.size[1]}) -> "
    f"{SHOWCASE_TARGET.name} ({showcase_target_img.size[0]}x{showcase_target_img.size[1]})"
)
show_images(
    [showcase_target_img],
    [f"Showcase target\n{showcase_target_img.size[0]}x{showcase_target_img.size[1]}"],
    max_side=300,
)

In [ ]:
# Guard: the full-gallery database is large (gitignored) and may not exist on every
# machine, or may predate a schema change (e.g. the fixture-detection columns added
# alongside tiler.periodicity_score). Either way, skip the rest of this section
# instead of failing.
SHOWCASE_AVAILABLE = False
if FULL_DB.exists():
    showcase_conn = open_database(FULL_DB)
    try:
        ensure_schema(showcase_conn)
        showcase_tile_sides = get_tile_sides(showcase_conn)
        SHOWCASE_AVAILABLE = True
        print(f"Full gallery database found: {FULL_DB.name} (tile sides: {showcase_tile_sides})")
    except SchemaVersionError as exc:
        print(
            f"NOTE: {FULL_DB} is out of date for this mosaic-builder version -- skipping the "
            f"showcase render. {exc}"
        )
    finally:
        showcase_conn.close()
else:
    print(
        f"NOTE: full gallery database not found at {FULL_DB} -- skipping the showcase render. "
        "Build it with: uv run mosaic-builder ingest gallery --db mosaic.duckdb --tile-side 64"
    )

In [ ]:
if SHOWCASE_AVAILABLE:
    # Roughly SHOWCASE_CELLS cells along the target's short side, whatever the target's shape.
    showcase_grain = max(8, min(showcase_target_img.size) // SHOWCASE_CELLS)

    showcase_result = run_match(
        SHOWCASE_TARGET,
        FULL_DB,
        grain=showcase_grain,
        diversity=SHOWCASE_DIVERSITY,
        show_progress=False,
    )
    print(summarize(showcase_result))

    showcase_conn = open_database(FULL_DB)
    ensure_schema(showcase_conn)
    try:
        showcase_mosaic_img = assemble_mosaic(showcase_result, showcase_conn, show_progress=False)
        showcase_gallery_images = len(get_all_images(showcase_conn))
        showcase_gallery_tiles = len(
            get_tile_lab_descriptors(showcase_conn, tile_side=showcase_result.tile_side)
        )
    finally:
        showcase_conn.close()

    print(f"Mosaic size: {showcase_mosaic_img.size[0]}x{showcase_mosaic_img.size[1]}")
    print(
        f"Gallery: {showcase_gallery_images} images, {showcase_gallery_tiles} tiles "
        f"at tile_side={showcase_result.tile_side}"
    )

In [ ]:
if SHOWCASE_AVAILABLE:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    # --- mosaic ---
    mosaic_path = OUTPUT_DIR / "urinal_mosaic.png"
    showcase_mosaic_img.save(mosaic_path, format="PNG")
    if mosaic_path.stat().st_size >= MOSAIC_CEILING_BYTES:
        showcase_mosaic_img.save(mosaic_path, format="PNG", optimize=True)
    assert mosaic_path.stat().st_size < MOSAIC_CEILING_BYTES, (
        f"{mosaic_path} is {mosaic_path.stat().st_size} bytes, over the {MOSAIC_CEILING_BYTES} byte ceiling"
    )

    # --- side-by-side comparison: target | gutter | mosaic, same height, capped width ---
    def _scaled(img, height):
        width = max(1, int(img.size[0] * height / img.size[1]))
        return img.resize((width, height), Image.Resampling.LANCZOS)

    comparison_height = 400
    gutter = 16
    target_scaled = _scaled(showcase_target_img, comparison_height)
    mosaic_scaled = _scaled(showcase_mosaic_img, comparison_height)
    total_width = target_scaled.size[0] + gutter + mosaic_scaled.size[0]
    if total_width > 1600:
        comparison_height = int(comparison_height * 1600 / total_width)
        target_scaled = _scaled(showcase_target_img, comparison_height)
        mosaic_scaled = _scaled(showcase_mosaic_img, comparison_height)
        total_width = target_scaled.size[0] + gutter + mosaic_scaled.size[0]

    comparison_img = Image.new("RGB", (total_width, comparison_height), (20, 20, 20))
    comparison_img.paste(target_scaled, (0, 0))
    comparison_img.paste(mosaic_scaled, (target_scaled.size[0] + gutter, 0))

    comparison_path = OUTPUT_DIR / "urinal_comparison.png"
    comparison_img.save(comparison_path, format="PNG")
    if comparison_path.stat().st_size >= COMPARISON_CEILING_BYTES:
        comparison_img.save(comparison_path, format="PNG", optimize=True)
    assert comparison_path.stat().st_size < COMPARISON_CEILING_BYTES, (
        f"{comparison_path} is {comparison_path.stat().st_size} bytes, over the "
        f"{COMPARISON_CEILING_BYTES} byte ceiling"
    )

    assert SHOWCASE_TARGET.stat().st_size < TARGET_CEILING_BYTES, (
        f"{SHOWCASE_TARGET} is {SHOWCASE_TARGET.stat().st_size} bytes, over the "
        f"{TARGET_CEILING_BYTES} byte ceiling"
    )

    # --- sidecar JSON ---
    showcase_mean_delta_e = (
        sum(m.delta_e for m in showcase_result.matches) / len(showcase_result.matches)
        if showcase_result.matches
        else 0.0
    )
    showcase_distinct_tiles = len({(m.image_id, m.tile_index) for m in showcase_result.matches})
    showcase_cli = (
        f"mosaic-builder mosaic {SHOWCASE_TARGET.relative_to(ROOT).as_posix()} "
        f"--db mosaic.duckdb --grain {showcase_grain} --tile-side {showcase_result.tile_side} "
        f"--diversity {SHOWCASE_DIVERSITY} "
        "--out examples/output/urinal_mosaic.png"
    )

    showcase_summary = {
        "target": SHOWCASE_TARGET.relative_to(ROOT).as_posix(),
        "target_size": list(showcase_target_img.size),
        "grain": showcase_grain,
        "tile_side": showcase_result.tile_side,
        "diversity": SHOWCASE_DIVERSITY,
        "blend": SHOWCASE_BLEND,
        "rows": showcase_result.rows,
        "cols": showcase_result.cols,
        "mosaic_size": list(showcase_mosaic_img.size),
        "mean_delta_e": round(showcase_mean_delta_e, 3),
        "distinct_tiles": showcase_distinct_tiles,
        "gallery_images": showcase_gallery_images,
        "gallery_tiles": showcase_gallery_tiles,
        "cli": showcase_cli,
    }

    json_path = OUTPUT_DIR / "urinal_showcase.json"
    json_path.write_text(json.dumps(showcase_summary, indent=2) + "\n")

    print(f"Wrote {mosaic_path.relative_to(ROOT)} ({mosaic_path.stat().st_size:,} bytes)")
    print(f"Wrote {comparison_path.relative_to(ROOT)} ({comparison_path.stat().st_size:,} bytes)")
    print(f"Wrote {json_path.relative_to(ROOT)}")

In [ ]:
if SHOWCASE_AVAILABLE:
    show_images(
        [comparison_img],
        [
            f"Target vs. mosaic ({showcase_summary['rows']}x{showcase_summary['cols']} grid, "
            f"{showcase_summary['distinct_tiles']} distinct tiles, "
            f"mean \u0394E {showcase_summary['mean_delta_e']})"
        ],
        max_side=1600,
    )
    print(json.dumps(showcase_summary, indent=2))

### Measured tradeoff on the real gallery

Measured on the full gallery (480 photos, 1440 tiles, tile_side 64) matched against the urinal
target at grain 25 — a 30×30 grid, 900 cells:

| `--diversity` | Mean ΔE | Max ΔE | Distinct tiles | Most-used tile |
|---|---|---|---|---|
| 0 | 3.24 | 5.21 | 76 | **138 cells** |
| 1 | 4.82 | 9.43 | 467 | 6 |
| 2 | 5.40 | 11.03 | 604 | 4 |
| 3 *(default)* | 5.77 | 12.18 | 699 | 3 |
| 5 | 6.40 | 12.97 | 815 | 2 |
| 7 | 6.70 | 13.77 | 856 | 2 |
| 10 | 7.04 | 16.27 | 889 | 2 |

The knee is sharp between 0 and 1: that first step costs 1.6 mean ΔE and buys 391 more distinct
tiles, collapsing the most-reused tile from 138 placements to 6. Everything after is diminishing
returns. The default of 3 keeps mean ΔE well inside the ~12 "looks off" threshold while capping any
single tile at 3 placements out of 900.

This replaced hard `--max-reuse`/`--min-repeat-dist` caps, which bought ~536 distinct tiles at a
mean ΔE of 13.92 — past the point where mismatches read as visibly wrong. Hard caps forced *every*
cell past its best tile once the limit hit, even where no comparable alternative existed; the soft
penalty only swaps where a near-equal tile is actually available. The gallery's tiles sit close
together in colour space (mean nearest-neighbour distance 1.06), so there is nearly always a
near-duplicate to swap to — which is exactly why the soft form wins here.

## Summary of decisions to evaluate

| Decision | What to look for |
|---|---|
| **Tile extraction** | Do the crop boxes cover the most interesting regions? Are boring areas avoided? |
| **Texture scoring** | Does the ranking match your intuition about visual interest? |
| **Fixture-detection scoring** | Do high `fixture_likelihood_score` tiles look fixture-adjacent? Do high `periodicity_score` tiles look like tiled/grid surfaces? |
| **Tile resize** | At tile_side=64, is enough detail preserved? Would 32 lose too much? |
| **Mean color swatches** | Does the swatch feel like a fair summary of the tile? |
| **LAB vs RGB distances** | Do the ΔE values better capture perceptual similarity than RGB distance? |
| **Tile diversity** | Is there enough spread in the ΔE matrix for a varied mosaic? |
| **Grid matching (`grain`)** | Does a coarser grain (bigger cells) blur the mosaic too much? Does a finer grain meaningfully improve it, or just slow matching down? |
| **Reuse / repeat-distance constraints** | Does forcing tile variety (lower `--max-reuse`, higher `--min-repeat-dist`) noticeably raise mean ΔE, per the measured tradeoff table above? Is the "~12 ΔE / ~300 distinct tiles" threshold a reasonable place to draw the line for this gallery? |
| **Rendering / blending** | Does the assembled mosaic read correctly at a distance? Does `blend_with_target` improve legibility, and at what alpha does it start to just look like the target again? |